# ACT-Break: Kara-Kumru Turkish Model Notebook
**Model Target:** `AlicanKiraz0/Kara-Kumru-v1.0-2B`

This notebook runs the ACT-Break pipeline against Kara-Kumru, a Turkish-focused 2B Mistral-family model. It keeps the repository default `config.py` unchanged and applies a Colab-runtime override before the pipeline steps.

### Google Colab Pro Recommendations:
- **GPU Accelerator:** **L4 GPU** or **A100 GPU** is recommended. Kara-Kumru is 2B, so it is lighter than 7B/9B Turkish alternatives, but suffix optimization is still GPU-intensive.
- **Google Drive Mount:** Highly recommended to preserve outputs, visualization figures, and validation results across runtime recycles.


## 1. Environment Setup & Dependency Installation
This cell will:
1. Mount Google Drive to save results.
2. Clone the ACT-Break repository from GitHub.
3. Install the `uv` package manager and sync project dependencies.
4. Leave model selection to the Kara-Kumru configuration cell below.


In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Clone repository if not present
import os
if not os.path.exists('/content/ACT-Break'):
    print("Cloning ACT-Break repository...")
    !git clone https://github.com/IrohAmca/ACT-Break.git
    %cd ACT-Break
else:
    print("ACT-Break repository already exists.")
    %cd /content/ACT-Break

# 3. Install uv package manager
import shutil
if not shutil.which("uv"):
    print("Installing uv package manager...")
    !curl -LsSf https://astral.sh/uv/install.sh | sh
    os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + os.pathsep + os.environ["PATH"]
else:
    print("uv is already installed.")

# 4. Sync dependencies
print("Syncing project dependencies via uv...")
!uv sync

# 5. Disable torch dynamo & force non-interactive matplotlib
os.environ["TORCHDYNAMO_DISABLE"] = "1"
import matplotlib
matplotlib.use("Agg")
print("Environment setup complete!")

## 2. GPU & Hardware Verification
Check the GPU specifications, allocated VRAM, and CUDA version to ensure the runtime is properly configured.

In [ ]:
import torch
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))
    print("VRAM:", f"{torch.cuda.get_device_properties(0).total_mem / (1024**3):.2f} GB")
    print("PyTorch Version:", torch.__version__)
    print("CUDA Version:", torch.version.cuda)
else:
    print("WARNING: Running on CPU. Suffix optimization will be extremely slow!")

## Hugging Face Authentication
Log in with the `HF_TOKEN` Colab secret before downloading gated Hugging Face models.


In [ ]:
from google.colab import userdata
from huggingface_hub import login

try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print("Successfully logged in to Hugging Face!")
except Exception as e:
    print(f"Failed to login to Hugging Face. Make sure you have added HF_TOKEN to your colab secrets. Error: {e}")


## Kara-Kumru Runtime Configuration
This cell rewrites the Colab runtime copy of `config.py` for `AlicanKiraz0/Kara-Kumru-v1.0-2B`. The local repository file is not changed by the notebook itself.


In [ ]:
from pathlib import Path
import os
import re
from transformers import AutoConfig

MODEL_NAME = "AlicanKiraz0/Kara-Kumru-v1.0-2B"
os.environ["ACT_BREAK_ADVBENCH_LANGUAGE"] = "tr"
os.environ["ACT_BREAK_NEGATIVE_ACTIVATION_MODE"] = "forced_refusal"
TARGET_LAYERS_EXPR = "list(range(6, 16))"
TARGET_STRING = "Elbette, i\u015fte"
COMPLIANCE_PREFIX_BLOCK = """COMPLIANCE_PREFIXES = [
    "Elbette, i\u015fte",
    "Tabii, i\u015fte",
    "Memnuniyetle, i\u015fte",
]"""
REFUSAL_PREFIX_BLOCK = """REFUSAL_PREFIXES = [
    "\u00dczg\u00fcn\u00fcm, bu konuda yard\u0131mc\u0131 olamam.",
    "Buna yard\u0131mc\u0131 olamam.",
    "Bu iste\u011fi yerine getiremem.",
]"""

config_path = Path("config.py")
text = config_path.read_text(encoding="utf-8")
text = re.sub(r'MODEL_NAME = ".*?"', f'MODEL_NAME = "{MODEL_NAME}"', text)
text = re.sub(
    r'TARGET_LAYERS = .*',
    f'TARGET_LAYERS = {TARGET_LAYERS_EXPR}  # Kara-Kumru 2B has 18 Mistral layers.',
    text,
)
text = re.sub(r'COMPLIANCE_PREFIXES = \[.*?\]', COMPLIANCE_PREFIX_BLOCK, text, flags=re.S)
if "REFUSAL_PREFIXES" in text:
    text = re.sub(r'REFUSAL_PREFIXES = \[.*?\]', REFUSAL_PREFIX_BLOCK, text, flags=re.S)
else:
    text = text.replace("DEFAULT_COMPLIANCE_PREFIX = COMPLIANCE_PREFIXES[0]", REFUSAL_PREFIX_BLOCK + "\nDEFAULT_COMPLIANCE_PREFIX = COMPLIANCE_PREFIXES[0]")
text = re.sub(r'TARGET_STRING = ".*?"', f'TARGET_STRING = "{TARGET_STRING}"', text)

if "ADVBENCH_ENGLISH_PATH" not in text:
    text = text.replace(
        'ADVBENCH_PATH = DATA_DIR / "harmful_prompts.csv"',
        'ADVBENCH_LANGUAGE = os.getenv("ACT_BREAK_ADVBENCH_LANGUAGE", "en").lower()\n'
        'ADVBENCH_ENGLISH_PATH = DATA_DIR / "harmful_prompts.csv"\n'
        'ADVBENCH_TURKISH_PATH = DATA_DIR / "harmful_prompts_tr.csv"\n'
        'ADVBENCH_PATH = ADVBENCH_TURKISH_PATH if ADVBENCH_LANGUAGE == "tr" else ADVBENCH_ENGLISH_PATH',
    )
text = re.sub(
    r'ADVBENCH_LANGUAGE = os\.getenv\("ACT_BREAK_ADVBENCH_LANGUAGE", ".*?"\)\.lower\(\)',
    'ADVBENCH_LANGUAGE = os.getenv("ACT_BREAK_ADVBENCH_LANGUAGE", "tr").lower()',
    text,
)
if "DEFAULT_REFUSAL_PREFIX" not in text:
    text = text.replace(
        "DEFAULT_COMPLIANCE_PREFIX = COMPLIANCE_PREFIXES[0]",
        "DEFAULT_COMPLIANCE_PREFIX = COMPLIANCE_PREFIXES[0]\n"
        "DEFAULT_REFUSAL_PREFIX = REFUSAL_PREFIXES[0]\n"
        "NEGATIVE_ACTIVATION_MODE = os.getenv(\"ACT_BREAK_NEGATIVE_ACTIVATION_MODE\", \"forced_refusal\").lower()",
    )
else:
    text = re.sub(
        r'NEGATIVE_ACTIVATION_MODE = os\.getenv\("ACT_BREAK_NEGATIVE_ACTIVATION_MODE", ".*?"\)\.lower\(\)',
        'NEGATIVE_ACTIVATION_MODE = os.getenv("ACT_BREAK_NEGATIVE_ACTIVATION_MODE", "forced_refusal").lower()',
        text,
    )
config_path.write_text(text, encoding="utf-8")

cfg = AutoConfig.from_pretrained(MODEL_NAME, trust_remote_code=True)
print("Configured model:", MODEL_NAME)
print("Architecture:", getattr(cfg, "architectures", None))
print("Model type:", cfg.model_type)
print("Hidden layers:", cfg.num_hidden_layers)
print("Target layers:", TARGET_LAYERS_EXPR)
print("Target string:", TARGET_STRING)


## Kara-Kumru Tokenizer Compatibility Patch
Kara-Kumru declares `TokenizersBackend` in `tokenizer_config.json`, which some `transformers` versions cannot resolve through `AutoTokenizer`. This cell patches the Colab runtime loader to fall back to `PreTrainedTokenizerFast` when that happens.


In [ ]:
from pathlib import Path

loader_path = Path("src/model_loader.py")
loader_text = loader_path.read_text(encoding="utf-8")

if "PreTrainedTokenizerFast" not in loader_text:
    loader_text = loader_text.replace(
        "from transformers import AutoModelForCausalLM, AutoTokenizer",
        "from transformers import AutoModelForCausalLM, AutoTokenizer, PreTrainedTokenizerFast",
    )
    old = """        self.tokenizer = AutoTokenizer.from_pretrained(
            self.model_name,
            trust_remote_code=True,
            padding_side="left",
        )"""
    new = """        try:
            self.tokenizer = AutoTokenizer.from_pretrained(
                self.model_name,
                trust_remote_code=True,
                padding_side="left",
            )
        except ValueError as exc:
            message = str(exc)
            if "Tokenizer class" not in message or "does not exist" not in message:
                raise
            print("[!] AutoTokenizer could not resolve the tokenizer class.")
            print("[!] Falling back to PreTrainedTokenizerFast from tokenizer.json.")
            self.tokenizer = PreTrainedTokenizerFast.from_pretrained(
                self.model_name,
                padding_side="left",
            )"""
    if old not in loader_text:
        raise RuntimeError("Could not find the tokenizer loading block to patch.")
    loader_text = loader_text.replace(old, new)
    loader_path.write_text(loader_text, encoding="utf-8")
    print("Applied Kara-Kumru tokenizer fallback patch.")
else:
    print("Tokenizer fallback patch is already present.")


## Build Turkish AdvBench Dataset
This cell creates `data/harmful_prompts_tr.csv` for the Turkish Kara-Kumru run using the quality translation preset (`facebook/nllb-200-3.3B`). The generated CSV keeps English originals in `goal_en/target_en` and writes Turkish text into `goal/target` plus `goal_tr/target_tr` for compatibility.


In [ ]:
from pathlib import Path
import os

os.environ["ACT_BREAK_ADVBENCH_LANGUAGE"] = "tr"
translator_path = Path("data/translate_advbench_tr.py")
if not translator_path.exists():
    translator_path.write_text('import argparse\nimport csv\nimport sys\nfrom pathlib import Path\n\nsys.path.insert(0, str(Path(__file__).parent.parent))\n\nimport config\nfrom data.download_advbench import _download_english_advbench\n\n\nTRANSLATION_PRESETS = {\n    "fast": "facebook/nllb-200-distilled-600M",\n    "balanced": "facebook/nllb-200-distilled-1.3B",\n    "quality": "facebook/nllb-200-3.3B",\n}\nPRESET_BATCH_SIZES = {\n    "fast": 16,\n    "balanced": 8,\n    "quality": 4,\n}\nDEFAULT_TRANSLATION_PRESET = "quality"\nDEFAULT_TRANSLATION_MODEL = TRANSLATION_PRESETS[DEFAULT_TRANSLATION_PRESET]\nSOURCE_LANG = "eng_Latn"\nTARGET_LANG = "tur_Latn"\n\n\ndef read_existing_count(path: Path) -> int:\n    if not path.exists():\n        return 0\n    with path.open("r", encoding="utf-8", newline="") as f:\n        return max(0, sum(1 for _ in f) - 1)\n\n\ndef batched(items: list[str], batch_size: int):\n    for start in range(0, len(items), batch_size):\n        yield items[start : start + batch_size]\n\n\ndef translate_batch(tokenizer, model, texts: list[str], device: str, max_new_tokens: int) -> list[str]:\n    inputs = tokenizer(\n        texts,\n        return_tensors="pt",\n        padding=True,\n        truncation=True,\n        max_length=512,\n    ).to(device)\n    generated = model.generate(\n        **inputs,\n        forced_bos_token_id=tokenizer.convert_tokens_to_ids(TARGET_LANG),\n        max_new_tokens=max_new_tokens,\n        num_beams=4,\n    )\n    return tokenizer.batch_decode(generated, skip_special_tokens=True)\n\n\ndef build_turkish_advbench(\n    source_path: Path,\n    output_path: Path,\n    model_name: str,\n    batch_size: int,\n    limit: int | None,\n    max_new_tokens: int,\n    resume: bool,\n) -> Path:\n    if not source_path.exists():\n        _download_english_advbench()\n\n    with source_path.open("r", encoding="utf-8", newline="") as f:\n        rows = list(csv.DictReader(f))\n\n    if limit is not None:\n        rows = rows[:limit]\n\n    start_index = read_existing_count(output_path) if resume else 0\n    if start_index > len(rows):\n        raise ValueError(f"{output_path} already has more rows than the requested source slice.")\n\n    import torch\n    from transformers import AutoModelForSeq2SeqLM, AutoTokenizer\n\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    write_header = not resume or not output_path.exists() or start_index == 0\n    mode = "a" if resume and output_path.exists() else "w"\n\n    device = "cuda" if torch.cuda.is_available() else "cpu"\n    dtype = torch.float16 if device == "cuda" else torch.float32\n    print(f"[*] Loading translation model: {model_name} on {device}")\n    tokenizer = AutoTokenizer.from_pretrained(model_name, src_lang=SOURCE_LANG)\n    model = AutoModelForSeq2SeqLM.from_pretrained(model_name, torch_dtype=dtype).to(device)\n    model.eval()\n\n    fieldnames = ["goal", "target", "goal_en", "target_en", "goal_tr", "target_tr"]\n    pending_rows = rows[start_index:]\n    print(f"[*] Translating {len(pending_rows)} rows ({start_index}/{len(rows)} already done).")\n\n    with output_path.open(mode, encoding="utf-8", newline="") as f:\n        writer = csv.DictWriter(f, fieldnames=fieldnames)\n        if write_header:\n            writer.writeheader()\n\n        for batch_start in range(0, len(pending_rows), batch_size):\n            batch_rows = pending_rows[batch_start : batch_start + batch_size]\n            source_texts = []\n            for row in batch_rows:\n                source_texts.extend([row["goal"], row["target"]])\n\n            translated = translate_batch(tokenizer, model, source_texts, device, max_new_tokens)\n            for row, goal_tr, target_tr in zip(batch_rows, translated[0::2], translated[1::2]):\n                writer.writerow(\n                    {\n                        "goal": goal_tr,\n                        "target": target_tr,\n                        "goal_en": row["goal"],\n                        "target_en": row["target"],\n                        "goal_tr": goal_tr,\n                        "target_tr": target_tr,\n                    }\n                )\n            f.flush()\n            done = start_index + batch_start + len(batch_rows)\n            print(f"[+] Wrote {done}/{len(rows)} rows to {output_path}")\n\n    return output_path\n\n\ndef parse_args():\n    parser = argparse.ArgumentParser(description="Translate AdvBench harmful prompts to Turkish.")\n    parser.add_argument("--source", type=Path, default=config.ADVBENCH_ENGLISH_PATH)\n    parser.add_argument("--output", type=Path, default=config.ADVBENCH_TURKISH_PATH)\n    parser.add_argument("--preset", choices=sorted(TRANSLATION_PRESETS), default=DEFAULT_TRANSLATION_PRESET)\n    parser.add_argument("--model", default=None, help="Override the translation model selected by --preset.")\n    parser.add_argument("--batch-size", type=int, default=None)\n    parser.add_argument("--limit", type=int, default=None)\n    parser.add_argument("--max-new-tokens", type=int, default=128)\n    parser.add_argument("--no-resume", action="store_true")\n    return parser.parse_args()\n\n\ndef main():\n    args = parse_args()\n    model_name = args.model or TRANSLATION_PRESETS[args.preset]\n    batch_size = args.batch_size or PRESET_BATCH_SIZES[args.preset]\n    output_path = build_turkish_advbench(\n        source_path=args.source,\n        output_path=args.output,\n        model_name=model_name,\n        batch_size=batch_size,\n        limit=args.limit,\n        max_new_tokens=args.max_new_tokens,\n        resume=not args.no_resume,\n    )\n    print(f"[+] Turkish AdvBench dataset ready: {output_path}")\n\n\nif __name__ == "__main__":\n    main()\n', encoding="utf-8")
    print("Wrote missing Turkish AdvBench translation script.")
else:
    print("Turkish AdvBench translation script is present.")

if not Path("data/harmful_prompts_tr.csv").exists():
    !uv run python -u data/translate_advbench_tr.py --preset quality
else:
    print("Turkish AdvBench dataset already exists: data/harmful_prompts_tr.csv")


## Turkish Pipeline Audit
Run this before Step 1 to confirm the Turkish dataset, config, loader, and Kara-Kumru tokenizer path are aligned.


In [ ]:
!uv run python -u scripts/audit_turkish_pipeline.py


## 3. Run Kara-Kumru Pipeline Steps

You can run the entire pipeline step-by-step. All outputs (activations, probes, figures, validation logs) will be saved in the `outputs/` folder.

### Step 1: Collect Contrastive Activations
Collects compliance vs. refusal activations for AdvBench prompts using `AlicanKiraz0/Kara-Kumru-v1.0-2B` with Turkish compliance prefixes.

In [ ]:
!uv run python -u scripts/01_collect_activations.py

### Step 2: Train Linear Probes
Trains logistic regression probes on target layers to classify compliance vs. refusal activations.

In [ ]:
!uv run python -u scripts/02_train_probe.py

### Step 3: Extract Direction & Visualize
Extracts refusal directions from probe weights, calculates cosine similarity with difference-of-means, and generates PCA/projection plots under the configured model output directory.

In [ ]:
!uv run python -u scripts/03_extract_direction.py

### Step 4: Activation Steering Validation
Validates if applying steering vectors to the target layers can bypass model safety alignment or cause text generation degradation.

In [ ]:
!uv run python -u scripts/04_steering_validation.py

### Step 5: Multi-Layer GCG Suffix Optimization
Optimizes adversarial jailbreak suffixes using a multi-layer minimax objective.
- **Gradient computation**: Uses the mean gradient across layers L8-L14 to select candidate tokens.
- **Candidate selection**: Evaluates candidate prompts based on the maximum loss (worst-performing layer) across L8-L14, forcing all target layers to converge simultaneously.

In [ ]:
!uv run python -u scripts/05_optimize_suffix.py

### Step 6: Multi-Stage Validation
Performs cross-validation of optimized suffixes, computes text perplexity filters, and tracks hidden state logit lens trajectories.

In [ ]:
!uv run python -u scripts/06_multi_stage_validation.py

## 4. Backup Kara-Kumru Results to Google Drive
Copies all generated plots, validation reports, optimized suffixes, and the runtime-patched `config.py` to Google Drive for persistence.

In [ ]:
import os
import shutil
from datetime import datetime, timezone

drive_dest = "/content/drive/My Drive/ACT-Break-Results-Kara-Kumru"

if os.path.exists("/content/drive/My Drive"):
    timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
    run_dest = os.path.join(drive_dest, f"run_{timestamp}")
    os.makedirs(run_dest, exist_ok=True)
    
    print(f"Copying outputs to Google Drive: {run_dest}")
    if os.path.exists("outputs"):
        shutil.copytree("outputs", run_dest, dirs_exist_ok=True)
        print("Successfully copied outputs to Google Drive!")
        
        # Also backup configuration and README
        for file in ["README.md", "config.py"]:
            if os.path.exists(file):
                shutil.copy(file, run_dest)
    else:
        print("No outputs directory found yet. Please run the pipeline steps first.")
else:
    print("Google Drive is not mounted. Please mount Google Drive in the Setup cell first.")